# A Priori Neural Network Correction for 1D Linear Advection

This notebook demonstrates the simplest instance of the corrective-term framework:
training a neural network to predict (and fix) the error introduced by a cheap
numerical scheme, using a fine-grid reference solution as ground truth.

**PDE:** $\quad u_t + c\,u_x = 0$ on a periodic domain.

**Setup:**
- **Fine grid:** $N_f = 400$ points, Lax-Wendroff scheme (2nd order, minimal diffusion)
- **Coarse grid:** $N_c = 50$ points, upwind scheme (1st order, diffusive)
- **NN correction:** learns $\delta_j = u_j^{\text{fine}} - u_j^{\text{coarse}}$ from local features

The coarse solver does the heavy lifting; the NN patches what it gets wrong.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

torch.manual_seed(42)
np.random.seed(42)

## 1. Parameters

The refinement ratio is $8\times$: the fine grid has 400 points, the coarse grid has 50.
The CFL number is 0.4 — well within stability for both schemes.

In [ ]:
L      = 1.0          # domain length (periodic)
c      = 1.0          # advection speed
N_f    = 400          # fine grid points
N_c    = 50           # coarse grid points (must divide N_f)
T      = 1.0          # simulation end time
CFL    = 0.4          # CFL number
N_ic   = 200          # number of initial conditions for training
N_test = 20           # held-out test ICs
EPOCHS = 300

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

ratio = N_f // N_c    # refinement ratio (8)

## 2. Grids and Time Steps

Both grids are uniform and periodic.  Each solver uses its own time step
sized to match the CFL number on its respective grid.

In [ ]:
x_f = np.linspace(0, L, N_f, endpoint=False)
x_c = np.linspace(0, L, N_c, endpoint=False)
dx_f = L / N_f
dx_c = L / N_c
dt_f = CFL * dx_f / c
dt_c = CFL * dx_c / c
N_steps_f = int(T / dt_f)
N_steps_c = int(T / dt_c)

print(f"Fine:   dx={dx_f:.4f}, dt={dt_f:.4f}, {N_steps_f} steps")
print(f"Coarse: dx={dx_c:.4f}, dt={dt_c:.4f}, {N_steps_c} steps")

## 3. Numerical Schemes

**Upwind (coarse):** first-order, introduces numerical diffusion.  The modified equation is
$$u_t + c\,u_x = \frac{c\,\Delta x}{2}(1-\sigma)\,u_{xx} + O(\Delta x^2)$$
so the scheme behaves like advection *plus artificial diffusion*.

**Lax-Wendroff (fine):** second-order, cancels the leading diffusion term.  The leading
error is dispersive ($u_{xxx}$), not diffusive — so the fine solution stays sharp.

In [ ]:
def upwind_step(u, c, dt, dx):
    """First-order upwind scheme (periodic BC)."""
    return u - c * dt / dx * (u - np.roll(u, 1))

def central_step(u, c, dt, dx):
    """Second-order Lax-Wendroff scheme (periodic BC)."""
    nu = c * dt / dx
    return u - nu/2 * (np.roll(u, -1) - np.roll(u, 1)) + nu**2/2 * (np.roll(u, -1) - 2*u + np.roll(u, 1))

def coarsen(u_fine, ratio):
    """Average fine solution onto coarse grid."""
    return u_fine.reshape(-1, ratio).mean(axis=1)

def simulate_fine(u0_f):
    u = u0_f.copy()
    for _ in range(N_steps_f):
        u = central_step(u, c, dt_f, dx_f)
    return u

def simulate_coarse(u0_c):
    u = u0_c.copy()
    for _ in range(N_steps_c):
        u = upwind_step(u, c, dt_c, dx_c)
    return u

## 4. The Problem: Numerical Diffusion

Before training anything, let's see what the upwind scheme does to a Gaussian pulse.
The fine solver preserves the shape; the coarse solver smears it out.

In [ ]:
# Single Gaussian test
u0_f = np.exp(-((x_f - 0.5)**2) / (2 * 0.06**2))
u0_c = coarsen(u0_f, ratio)

u_fine_T   = simulate_fine(u0_f)
u_coarse_T = simulate_coarse(u0_c)
u_fine_c   = coarsen(u_fine_T, ratio)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(x_c, u_fine_c,   'k-',  lw=2,   label='Fine (truth)')
ax.plot(x_c, u_coarse_T, 'r--', lw=1.8, label='Coarse (upwind)')
ax.set_title(f'1D Advection at T={T} — the smearing problem')
ax.set_xlabel('x'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"RMSE of coarse vs fine: {np.mean((u_fine_c - u_coarse_T)**2)**0.5:.4e}")

## 5. Training Data: Random Initial Conditions

We generate 220 random ICs (200 train, 20 test), alternating between Gaussian pulses
and sums of sinusoids.  Variety in the training set helps the network generalise —
it should learn to correct the *scheme's error*, not memorise specific solutions.

In [ ]:
def random_gaussian_ic(x, rng):
    """Gaussian pulse with random centre and width."""
    mu    = rng.uniform(0.2, 0.8)
    sigma = rng.uniform(0.03, 0.12)
    return np.exp(-((x - mu)**2) / (2 * sigma**2))

def random_multiscale_ic(x, rng):
    """Sum of a few sinusoids with random phases (smooth, periodic)."""
    u = np.zeros_like(x)
    for k in rng.choice(range(1, 6), size=3, replace=False):
        phi = rng.uniform(0, 2*np.pi)
        amp = rng.uniform(0.2, 1.0)
        u += amp * np.sin(2*np.pi*k*x/L + phi)
    return u / (np.max(np.abs(u)) + 1e-8)

rng = np.random.default_rng(0)

## 6. Build the Dataset

For each IC, we:
1. Run the fine solver → $u^f(T)$
2. Run the coarse solver → $u^c(T)$
3. Coarsen the fine solution → $\bar{u}^f(T)$
4. Compute features: $[u^c_j,\; du^c/dx,\; d^2u^c/dx^2]$ at each coarse point
5. Compute target: $\delta_j = \bar{u}^f_j - u^c_j$

The features are chosen because the modified equation tells us the leading error
depends on $u_{xx}$ — so giving the network access to local derivatives makes sense.

In [ ]:
print("Generating training dataset ...")

X_list, Y_list = [], []

for i in range(N_ic + N_test):
    if i % 2 == 0:
        u0_f = random_gaussian_ic(x_f, rng)
    else:
        u0_f = random_multiscale_ic(x_f, rng)

    u0_c = coarsen(u0_f, ratio)

    u_fine_T   = simulate_fine(u0_f)
    u_coarse_T = simulate_coarse(u0_c)

    u_fine_coarsened = coarsen(u_fine_T, ratio)

    # Input features: [u, du/dx, d²u/dx²] on coarse grid
    du_dx  = (np.roll(u_coarse_T, -1) - np.roll(u_coarse_T, 1)) / (2*dx_c)
    d2u_dx = (np.roll(u_coarse_T, -1) - 2*u_coarse_T + np.roll(u_coarse_T, 1)) / dx_c**2

    features = np.stack([u_coarse_T, du_dx, d2u_dx], axis=1)  # (N_c, 3)
    target   = u_fine_coarsened - u_coarse_T                   # correction (N_c,)

    X_list.append(features)
    Y_list.append(target)

X_all = np.array(X_list, dtype=np.float32)   # (220, 50, 3)
Y_all = np.array(Y_list, dtype=np.float32)   # (220, 50)

# Flatten: treat each grid point as an independent sample
X_train = X_all[:N_ic].reshape(-1, 3)
Y_train = Y_all[:N_ic].reshape(-1)
X_test  = X_all[N_ic:].reshape(-1, 3)
Y_test  = Y_all[N_ic:].reshape(-1)

X_train_t = torch.tensor(X_train).to(DEVICE)
Y_train_t = torch.tensor(Y_train).unsqueeze(1).to(DEVICE)
X_test_t  = torch.tensor(X_test).to(DEVICE)
Y_test_t  = torch.tensor(Y_test).unsqueeze(1).to(DEVICE)

print(f"Training samples: {X_train_t.shape[0]}  (= {N_ic} ICs × {N_c} points)")
print(f"Test samples:     {X_test_t.shape[0]}")

## 7. The Correction Network

A small MLP: 3 inputs → 64 → 64 → 64 → 1 output, with Tanh activations.

Tanh is chosen because the correction can be positive or negative (unlike ReLU which
is zero for negative inputs). The network is intentionally small — the mapping from
local features to correction is smooth and low-dimensional.

In [ ]:
class CorrectionNet(nn.Module):
    def __init__(self, in_features=3, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),      nn.Tanh(),
            nn.Linear(hidden, hidden),      nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x)

model = CorrectionNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
loss_fn = nn.MSELoss()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print(model)

## 8. Training

Full-batch training with Adam and cosine-annealing learning rate schedule.
The cosine schedule smoothly decays the learning rate from $10^{-3}$ to near zero
over 300 epochs — large steps early for fast progress, small steps late to settle
into the minimum.

In [ ]:
print("Training correction network ...")
train_losses, test_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train_t)
    loss = loss_fn(pred, Y_train_t)
    loss.backward()
    optimizer.step()
    scheduler.step()

    model.eval()
    with torch.no_grad():
        test_loss = loss_fn(model(X_test_t), Y_test_t).item()
    train_losses.append(loss.item())
    test_losses.append(test_loss)

    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1:4d}  train={loss.item():.2e}  test={test_loss:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(train_losses, label='Train', alpha=0.7)
ax.semilogy(test_losses,  label='Test',  alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Training convergence')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Evaluation on a Fresh Initial Condition

We test on a Gaussian pulse that was **not** in the training set.  The network
predicts the correction from the coarse solution's local features, and the
corrected solution is simply:

$$u^{\text{corrected}}_j = u^{\text{coarse}}_j + \mathcal{N}_\theta(u^{\text{coarse}}_j,\; du/dx,\; d^2u/dx^2)$$

In [ ]:
model.eval()

rng2 = np.random.default_rng(999)
u0_f = random_gaussian_ic(x_f, rng2)
u0_c = coarsen(u0_f, ratio)

u_fine_T    = simulate_fine(u0_f)
u_coarse_T  = simulate_coarse(u0_c)
u_fine_c    = coarsen(u_fine_T, ratio)

du_dx  = (np.roll(u_coarse_T, -1) - np.roll(u_coarse_T, 1)) / (2*dx_c)
d2u_dx = (np.roll(u_coarse_T, -1) - 2*u_coarse_T + np.roll(u_coarse_T, 1)) / dx_c**2
feat   = np.stack([u_coarse_T, du_dx, d2u_dx], axis=1).astype(np.float32)

with torch.no_grad():
    delta_pred = model(torch.tensor(feat).to(DEVICE)).cpu().numpy().flatten()

u_corrected = u_coarse_T + delta_pred

## 10. Results

**Left:** the coarse solution (red, smeared) vs the fine reference (black, sharp) vs
the corrected solution (blue).  The NN correction recovers most of the lost sharpness.

**Right:** pointwise error before and after correction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_c, u_fine_c,    'k-',  lw=2,   label='Fine (truth)')
ax.plot(x_c, u_coarse_T,  'r--', lw=1.8, label='Coarse (upwind)')
ax.plot(x_c, u_corrected, 'b-',  lw=1.8, label='Coarse + NN correction')
ax.set_title(f'1D Advection at T={T:.2f}')
ax.set_xlabel('x')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_c, u_fine_c - u_coarse_T,  'r--', label='Error without correction')
ax.plot(x_c, u_fine_c - u_corrected, 'b-',  label='Error with NN correction')
ax.axhline(0, color='k', lw=0.8)
ax.set_title('Pointwise error')
ax.set_xlabel('x')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

err_before = np.mean((u_fine_c - u_coarse_T)**2)**0.5
err_after  = np.mean((u_fine_c - u_corrected)**2)**0.5
print(f"RMSE before correction: {err_before:.4e}")
print(f"RMSE after  correction: {err_after:.4e}")
print(f"Improvement factor:     {err_before/err_after:.1f}x")

## 10b. Stress Test: Square Pulse

The training data contained only smooth ICs (Gaussians and sinusoids).  A square pulse
is **out of distribution** — it has discontinuities that produce infinite derivatives,
which the network has never seen.

This is a useful diagnostic: does the network degrade gracefully on inputs it wasn't
trained for, or does it make things worse?  The Lax-Wendroff fine solver itself will
produce Gibbs oscillations near the edges of the square, so the "truth" isn't perfect
either — but the coarse upwind scheme will heavily smear the pulse.

In [ ]:
# Square pulse: u = 1 on [0.25, 0.55], 0 elsewhere
u0_sq_f = np.where((x_f > 0.25) & (x_f < 0.55), 1.0, 0.0)
u0_sq_c = coarsen(u0_sq_f, ratio)

u_sq_fine_T   = simulate_fine(u0_sq_f)
u_sq_coarse_T = simulate_coarse(u0_sq_c)
u_sq_fine_c   = coarsen(u_sq_fine_T, ratio)

# NN correction
du_dx_sq  = (np.roll(u_sq_coarse_T, -1) - np.roll(u_sq_coarse_T, 1)) / (2*dx_c)
d2u_dx_sq = (np.roll(u_sq_coarse_T, -1) - 2*u_sq_coarse_T + np.roll(u_sq_coarse_T, 1)) / dx_c**2
feat_sq   = np.stack([u_sq_coarse_T, du_dx_sq, d2u_dx_sq], axis=1).astype(np.float32)

with torch.no_grad():
    delta_sq = model(torch.tensor(feat_sq).to(DEVICE)).cpu().numpy().flatten()

u_sq_corrected = u_sq_coarse_T + delta_sq

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_c, u_sq_fine_c,      'k-',  lw=2,   label='Fine (truth)')
ax.plot(x_c, u_sq_coarse_T,    'r--', lw=1.8, label='Coarse (upwind)')
ax.plot(x_c, u_sq_corrected,   'b-',  lw=1.8, label='Coarse + NN correction')
ax.set_title(f'Square pulse at T={T:.2f} (out of distribution)')
ax.set_xlabel('x')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_c, u_sq_fine_c - u_sq_coarse_T,    'r--', label='Error without correction')
ax.plot(x_c, u_sq_fine_c - u_sq_corrected,   'b-',  label='Error with NN correction')
ax.axhline(0, color='k', lw=0.8)
ax.set_title('Pointwise error (square pulse)')
ax.set_xlabel('x')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

err_sq_before = np.mean((u_sq_fine_c - u_sq_coarse_T)**2)**0.5
err_sq_after  = np.mean((u_sq_fine_c - u_sq_corrected)**2)**0.5
print(f"Square pulse RMSE before: {err_sq_before:.4e}")
print(f"Square pulse RMSE after:  {err_sq_after:.4e}")
if err_sq_after < err_sq_before:
    print(f"Improvement: {err_sq_before/err_sq_after:.1f}x — network helps even out of distribution")
else:
    print(f"Degradation: {err_sq_after/err_sq_before:.2f}x — network hurts on this OOD input")
print("\nThis is typical of a priori training: the correction helps on inputs similar to")
print("the training data but can struggle or misbehave on very different inputs.")

## 11. What Did the Network Learn?

We can probe the network by sweeping one input while holding the others at zero.
Since the modified equation says the leading error is proportional to $u_{xx}$,
we expect the network's response to the second-derivative input to be roughly linear.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
feature_names = ['$u$', '$du/dx$', '$d^2u/dx^2$']

for i, (ax, name) in enumerate(zip(axes, feature_names)):
    sweep = np.linspace(-2, 2, 200)
    inp = np.zeros((200, 3), dtype=np.float32)
    inp[:, i] = sweep
    with torch.no_grad():
        out = model(torch.tensor(inp).to(DEVICE)).cpu().numpy().flatten()
    ax.plot(sweep, out, 'b-', lw=1.5)
    print(out.min(), out.max(), out.max()-out.min())
    ax.set_xlabel(name); ax.set_ylabel('NN output')
    ax.set_title(f'Response to {name}')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_ylim([-.5,.5])

plt.suptitle('Network response to each input feature (others = 0)', fontsize=12)
plt.tight_layout()
plt.show()